In [1]:
#Installing some dependencies
!pip install -q transformers accelerate anthropic openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 36.5 MB/s eta 0:00:00


In [2]:
#Imports

import os
import torch
from google.colab import userdata
from transformers import pipeline
import anthropic
from openai import OpenAI
import gradio as gr
from huggingface_hub import login
from IPython.display import Markdown, display

In [3]:
# Setting some environment variables
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["ANTHROPIC_API_KEY"] = userdata.get('ANTHROPIC_API_KEY')

device = "cuda" if torch.cuda.is_available() else "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Using device: {device}")


Using device: cuda


In [4]:
# Signing to Huggingface
hf_token = userdata.get("HF_TOKEN")
if hf_token is None:
  raise ValueError

login(hf_token, add_to_git_credential=True)

In [5]:
# Load Whisper (HuggingFace Pipeline)

transcriber = pipeline(
    task="automatic-speech-recognition",
    model="openai/whisper-large-v3",
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True
)
print("whisper loaded successfully...")

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

...

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'return_timestamps'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


whisper loaded successfully...


In [6]:
# Initialize Claude and OpenAI clients

claude = anthropic.Anthropic()
openai_client = OpenAI()
print("Claude and OpenAI clients initialized successfully...")

Claude and OpenAI clients initialized successfully...


In [7]:
SYSTEM_PROMPT = """You are an expert music critic and literary analyst. Given song lyrics, provide a deep, thoughtful analysis.

Structure your response in Markdown with these sections:

## Themes & Subject Matter
What is this song really about? Core ideas explored.

## Emotional Tone
The dominant emotions and atmosphere conveyed.

## Literary Devices
Metaphors, symbolism, imagery, wordplay used in the lyrics.

## Hidden Meanings
Subtext, deeper interpretations, possible biographical or cultural context.

## Mood & Visual Description
A vivid sensory description that could be translated into artwork.

At the very end, on a new line write exactly:
COVER_PROMPT: [a detailed visual prompt for AI image generation — focus on mood, colors, art style, key imagery, lighting. No text or letters in the image.]

Be insightful, articulate, and avoid clichés."""

def analyze_lyrics(lyrics):
    response = openai_client.chat.completions.create(
        model="gpt-4o", # Using a powerful OpenAI model
        max_tokens=2000,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Analyze these song lyrics:\n\n{lyrics}"}
        ]
    )
    return response.choices[0].message.content

In [8]:
# Cover Art/Image Generation

def generate_cover_art(prompt):
  response = openai_client.images.generate(
      model="dall-e-2", # Changed from dall-e-3 to dall-e-2
      prompt=f"Album cover art:{prompt}. Artistic, evocative, professional album cover design. No text, no words, no letters.",
      size="1024x1024",
      n=1,
  )
  return response.data[0].url

In [14]:
# Main Pipeline

def process_song(audio_file):
    try:
        print("Function started")
        if audio_file is None:
            print("No file uploaded")
            yield "⚠️ Please upload an audio file.", "", None
            return
        print("Starting transcription")

        yield "🎙️ Transcribing lyrics with Whisper...", "", None

        transcription = transcriber(audio_file)
        print("Transcription done")

        lyrics = transcription["text"]

        yield "🔍 Analyzing lyrics...", "", None

        print("Starting analysis")

        analysis = analyze_lyrics(lyrics)

        print("Analysis completed")
        print(analysis)

        if "COVER_PROMPT" in analysis:
            print("Cover prompt found")

            cover_prompt = analysis.split("COVER_PROMPT:")[-1].strip()

            display_analysis = analysis.split("COVER_PROMPT:")[0].strip()

        else:
            print("Using default cover prompt")

            cover_prompt = "abstract artistic visualization of music and emotion"

            display_analysis = analysis

        yield lyrics, display_analysis, None

        print("Generating image")

        image_url = generate_cover_art(cover_prompt)

        print("Image generated")
        print(image_url)

        yield lyrics, display_analysis, image_url

    except Exception as e:
        print("FULL ERROR:", e)

        yield f"Error: {str(e)}", "", None


In [ ]:
# Gradio UI

with gr.Blocks(title="AI Song Lyrics Analyzer", theme=gr.themes.Soft()) as app:
    gr.Markdown("# 🎵 AI Song Lyrics Analyzer")
    gr.Markdown(
        "Upload a song → transcribes lyrics → analyzes meaning → creates cover art"
    )

    audio_input = gr.Audio(type="filepath", label="🎧 Upload Your Song (mp3, wav, m4a)")
    analyze_btn = gr.Button("✨ Analyze Song", variant="primary", size="lg")

    with gr.Row():
        with gr.Column(scale=1):
            lyrics_output = gr.Textbox(
                label="📝 Transcribed Lyrics",
                lines=10,
                show_copy_button=True
            )
            analysis_output = gr.Markdown(label="🔬 Deep Analysis")
        with gr.Column(scale=1):
            cover_output = gr.Image(label="🎨 AI-Generated Cover Art", height=500)

    analyze_btn.click(
        process_song,
        inputs=[audio_input],
        outputs=[lyrics_output, analysis_output, cover_output]
    )

app.launch(share=True, debug=True)


/tmp/ipykernel_2407/13478424.py:3: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="AI Song Lyrics Analyzer", theme=gr.themes.Soft()) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://66807aada50eb6ee2b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
